# Riforces E-Commerce — Istanbul Delivery Network Optimization
### MIS Project | Shortest Path Analysis (Dijkstra's Algorithm)

This notebook walks through the full analysis step by step:
1. Load and explore the network data
2. Build the graph
3. Solve shortest paths (Min Time)
4. Visualize the network
5. Interpret results managerially

## 1. Import Libraries

In [ ]:
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

print('Libraries loaded successfully!')

## 2. Load & Explore the Dataset

In [ ]:
df = pd.read_csv('../data/network_data.csv')
print(f'Total edges: {len(df)}')
df

In [ ]:
print('=== Basic Statistics ===')
df[['distance_km','cost_usd','time_hours']].describe().round(2)

In [ ]:
# Unique nodes
all_nodes = pd.concat([df['source'], df['target']]).unique()
print(f'Total unique nodes: {len(all_nodes)}')
for n in sorted(all_nodes):
    ntype = 'DC' if 'DC_' in n else ('Hub' if 'Hub_' in n else 'Zone')
    print(f'  [{ntype}]  {n}')

## 3. Build the Network Graph

In [ ]:
df_clean = df.drop_duplicates(subset=['source','target'], keep='first')

G = nx.DiGraph()
for _, row in df_clean.iterrows():
    G.add_edge(row['source'], row['target'],
               cost_usd=row['cost_usd'],
               time_hours=row['time_hours'],
               distance_km=row['distance_km'],
               weight=row['time_hours'])  # Primary: minimize time

print(f'Graph created: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

## 4. Solve: Shortest Path (Minimum Time — Dijkstra)

In [ ]:
SOURCE = 'DC_Riforces'
lengths, paths = nx.single_source_dijkstra(G, source=SOURCE, weight='weight')

zones = sorted(k for k in lengths if k.startswith('Zone_'))

print(f'Optimal routes from {SOURCE}:\n')
print(f'{"Zone":<25} {"Time(h)":<10} {"Cost($)":<10} {"Dist(km)":<12} Route')
print('-'*85)
for zone in zones:
    path = paths[zone]
    cost  = sum(G[path[i]][path[i+1]]['cost_usd']    for i in range(len(path)-1))
    time_ = sum(G[path[i]][path[i+1]]['time_hours']   for i in range(len(path)-1))
    dist  = sum(G[path[i]][path[i+1]]['distance_km']  for i in range(len(path)-1))
    print(f'{zone:<25} {round(time_,2):<10} {round(cost,2):<10} {round(dist,1):<12} {" → ".join(path)}')

## 5. Network Visualization

In [ ]:
pos = {
    'DC_Riforces':       (0.50, 0.88),
    'Hub_Sisli':         (0.33, 0.67),
    'Hub_Besiktas':      (0.20, 0.55),
    'Hub_Kadikoy':       (0.66, 0.58),
    'Hub_Bakirkoy':      (0.24, 0.37),
    'Zone_Levent':       (0.15, 0.74),
    'Zone_Sariyer':      (0.06, 0.60),
    'Zone_Atasehir':     (0.80, 0.45),
    'Zone_Maltepe':      (0.73, 0.29),
    'Zone_Pendik':       (0.91, 0.18),
    'Zone_Avcilar':      (0.13, 0.21),
    'Zone_Buyukcekmece': (0.02, 0.10),
}

optimal_edges = set()
for zone in zones:
    p = paths[zone]
    for i in range(len(p)-1):
        optimal_edges.add((p[i], p[i+1]))

fig, ax = plt.subplots(figsize=(16, 11))
fig.patch.set_facecolor('#0f1923')
ax.set_facecolor('#0f1923')

node_colors, node_sizes = [], []
for node in G.nodes():
    if 'DC_' in node:     node_colors.append('#f97316'); node_sizes.append(2200)
    elif 'Hub_' in node:  node_colors.append('#3b82f6'); node_sizes.append(1500)
    else:                 node_colors.append('#22c55e'); node_sizes.append(1000)

edge_colors, edge_widths = [], []
for u, v in G.edges():
    if (u, v) in optimal_edges:
        edge_colors.append('#facc15'); edge_widths.append(3.5)
    else:
        edge_colors.append('#475569'); edge_widths.append(1.2)

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, ax=ax, alpha=0.95)
nx.draw_networkx_edges(G, pos, edge_color=edge_colors, width=edge_widths,
                       ax=ax, arrows=True, arrowsize=18,
                       connectionstyle='arc3,rad=0.08', alpha=0.85)

labels = {n: n.replace('DC_Riforces','DC\nRiforces').replace('Hub_','').replace('Zone_','') for n in G.nodes()}
nx.draw_networkx_labels(G, pos, labels=labels, font_size=8, font_color='white', font_weight='bold', ax=ax)

edge_labels = {(u,v): '$' + str(d['cost_usd']) + ' | ' + str(d['time_hours']) + 'h' for u,v,d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels,
                              font_size=7, font_color='#94a3b8', ax=ax,
                              bbox=dict(boxstyle='round,pad=0.2', fc='#1e293b', alpha=0.7))

legend_handles = [
    mpatches.Patch(color='#f97316', label='Distribution Center (DC)'),
    mpatches.Patch(color='#3b82f6', label='Hub Warehouse'),
    mpatches.Patch(color='#22c55e', label='Customer Delivery Zone'),
    mpatches.Patch(color='#facc15', label='Optimal Route (Min Time)'),
    mpatches.Patch(color='#475569', label='Available Route'),
]
ax.legend(handles=legend_handles, loc='lower right', facecolor='#1e293b',
          edgecolor='#334155', labelcolor='white', fontsize=9)

ax.set_title('Riforces E-Commerce | Istanbul Delivery Network\nShortest Path — Min Time  |  Edge: $Cost | Time(h)',
             color='white', fontsize=14, fontweight='bold', pad=15)
ax.axis('off')
plt.tight_layout()
plt.show()

## 6. Managerial Interpretation

### Key Findings

| Finding | Detail |
|---|---|
| **Fastest Zone** | Zone_Levent — 0.15h via DC → Hub_Sisli → Zone_Levent |
| **Slowest Zone** | Zone_Buyukcekmece — 1.75h, high cost ($35) |
| **Critical Hub** | Hub_Kadikoy serves 3 zones — single point of failure |
| **Best Hub** | Hub_Sisli — lowest cost & time for western zones |

### Strategic Recommendations

1. **Prioritize Hub_Sisli capacity** — it serves the most cost/time-efficient corridor
2. **Add failover routing** at Hub_Kadikoy via Hub_Sisli as backup
3. **Apply surcharge** for zones beyond 40 km (Buyukcekmece, Pendik)
4. **Consider a micro-hub** near Atasehir to cut eastern delivery time by ~0.3h